# Practical 2 — Stopword Removal

**Course:** NLP
**Name:**  <!-- fill in -->
**Date:**  <!-- fill in -->

## Aim
To remove stopwords from the cleaned/tokenized review corpus using NLTK's default English stopword list, and to compare that against a custom domain-specific stopword list — including checking what stopword removal does to negation words.

## Theory

**Stopwords** are high-frequency words (articles, prepositions, common pronouns/verbs — "the", "is", "at", "which") that appear in almost every document and typically carry little topic-specific meaning on their own. Removing them before tasks like classification or topic modelling reduces vocabulary size and lets the model focus on words that actually discriminate between documents.

Two important caveats, though:
- **Stopword lists are task-dependent.** NLTK's general English list is built for broad use, not for any specific domain. A movie-review corpus might have its own high-frequency, low-signal words ("movie", "film", "watch") that aren't in NLTK's list but behave like stopwords *for this dataset*.
- **Stopword removal can delete meaningful signal**, especially for sentiment analysis. NLTK's default list includes negation-related words like `"not"`, `"no"`, `"nor"`, and contraction pieces like `"wasn"`, `"wouldn"`, `"shouldn"`. Removing these can flip or erase the sentiment of a sentence — `"not good"` losing "not" leaves just `"good"`, which is the opposite meaning. This is a direct continuation of the tokenization practical: how a contraction gets tokenized upstream affects whether stopword removal even recognizes it downstream.

## Algorithm

1. Reuse the cleaned + tokenized reviews from Practical 1 (`preprocessing.clean_text` + `tokenizer.regex_word_tokenize`).
2. Load NLTK's default English stopword list and check its size.
3. Remove default stopwords from the tokens; compare before/after for a few reviews.
4. Specifically inspect what happened to any negation-related words during cleaning and stopword removal.
5. Load the custom domain stopword list (`datasets/stopwords_custom.txt`) and remove those too.
6. Recompute vocabulary size after each stage (raw tokens → default stopwords removed → custom stopwords also removed) and compare to Practical 1's baseline (136 unique tokens).

In [ ]:
import sys, os
sys.path.append(os.path.abspath("../python"))

import pandas as pd
import nltk

nltk.download("stopwords")

import preprocessing
import tokenizer

pd.set_option("display.max_colwidth", None)

df = pd.read_csv("../datasets/sample_reviews.csv")
df["tokens"] = df["review"].apply(lambda r: tokenizer.regex_word_tokenize(preprocessing.clean_text(r)))
print(f"Loaded {len(df)} reviews")


### Step 1 — NLTK's default stopword list

In [ ]:
nltk_stops = preprocessing.get_nltk_stopwords()
print(f"NLTK stopword list size: {len(nltk_stops)}")
print(sorted(nltk_stops)[:30])


### Step 2 — Remove default stopwords: before vs after

In [ ]:
for i in range(3):
    original = df.loc[i, "tokens"]
    filtered = preprocessing.remove_stopwords(original, nltk_stops)
    print(f"BEFORE ({len(original)} tokens): {original}")
    print(f"AFTER  ({len(filtered)} tokens): {filtered}")
    print("-" * 60)


### Step 3 — Check what happened to negation words specifically

Look for any of NLTK's negation-related stopwords (`not`, `no`, `nor`, and contraction pieces like `wasn`, `wouldn`, `shouldn`, `isn`, `ain`) in the tokens *before* stopword removal — then check whether they actually got removed, and whether they were even present in that form after Practical 1's cleaning step stripped apostrophes.

In [ ]:
negation_related = {"not", "no", "nor", "wasn", "wouldn", "shouldn", "isn", "ain", "don", "couldn", "doesn", "didn"}

for i, row in df.iterrows():
    found = [t for t in row["tokens"] if t in negation_related]
    if found:
        print(f"Review {row['id']}: found {found} in tokens -> {row['tokens']}")


### Step 4 — Layer on the custom domain stopword list

In [ ]:
custom_stops = preprocessing.load_custom_stopwords("../datasets/stopwords_custom.txt")
print(f"Custom stopword list: {sorted(custom_stops)}")

combined_stops = nltk_stops | custom_stops

df["tokens_no_default_stops"] = df["tokens"].apply(lambda t: preprocessing.remove_stopwords(t, nltk_stops))
df["tokens_no_combined_stops"] = df["tokens"].apply(lambda t: preprocessing.remove_stopwords(t, combined_stops))

for i in range(2):
    print(f"RAW:              {df.loc[i, 'tokens']}")
    print(f"NLTK STOPS OUT:   {df.loc[i, 'tokens_no_default_stops']}")
    print(f"+ CUSTOM STOPS:   {df.loc[i, 'tokens_no_combined_stops']}")
    print("-" * 60)


### Step 5 — Vocabulary size at each stage

In [ ]:
def vocab_size(token_lists):
    return len(set(t for tokens in token_lists for t in tokens))

raw_vocab = vocab_size(df["tokens"])
nltk_filtered_vocab = vocab_size(df["tokens_no_default_stops"])
combined_filtered_vocab = vocab_size(df["tokens_no_combined_stops"])

print(f"Raw vocabulary (Practical 1 baseline):        {raw_vocab}")
print(f"After NLTK stopword removal:                  {nltk_filtered_vocab}")
print(f"After NLTK + custom domain stopword removal:  {combined_filtered_vocab}")


---
## Output

*Run every cell above top to bottom, then paste or describe your actual output here — the NLTK stopword list size, the before/after examples, what you found (or didn't find) when checking for negation words, and the three vocabulary size numbers. Don't fill this in until you've actually run it.*


## Observations & Conclusion

Answer these based on what you actually saw when you ran the notebook:

- Did any negation-related stopwords actually show up and get removed from your tokens — or did Practical 1's apostrophe-stripping already change contractions into a form that doesn't match NLTK's stopword list? What does that imply about doing preprocessing steps in the wrong order?
- How much did vocabulary size shrink after NLTK stopword removal, and then again after adding the custom domain list? Which cut more?
- Do you think stopword removal was net positive or net risky for this specific dataset, if the eventual goal were sentiment analysis?

*(Write 4-6 sentences here in your own words once you've run the notebook.)*


---
## Viva Prep — Practice Questions

1. **What is a stopword, and why are they usually removed before text classification?**
   A stopword is a high-frequency word that carries little document-specific meaning (e.g. "the", "is", "at"). Removing them shrinks the vocabulary and lets models focus on words that actually help distinguish between documents/classes.

2. **Why might a general-purpose stopword list be a bad fit for a specific domain?**
   Because "high-frequency but low-signal" is domain-relative — words like "movie" or "film" are uninformative in a movie-review corpus specifically, but wouldn't appear in a general English stopword list built for broad use.

3. **Why can stopword removal be risky for sentiment analysis specifically?**
   Because common stopword lists include negation words ("not", "no", "nor") and contraction fragments ("wasn", "wouldn"). Removing "not" from "not good" leaves "good" — flipping the apparent sentiment of the sentence.

4. **Why does the order of preprocessing steps matter here?**
   If contractions get stripped of their apostrophes (e.g. "wasn't" -> "wasnt") before stopword removal runs, the result won't match stopword list entries like "wasn", so the negation word silently survives in a mangled form instead of being either cleanly kept or intentionally removed — an unintended side effect of step ordering.

5. **What's the tradeoff of adding a custom domain stopword list on top of a general one?**
   It further reduces vocabulary size and noise for that specific dataset, but risks removing words that could matter in a different context (e.g. "watch" might be irrelevant for a movie review's sentiment, but could matter if the task were topic classification, e.g. distinguishing reviews from watch/product ads).
